# WarehousePG Demo: Attribute-Based Access Control (ABAC)¶

A tag-based attribute-based access control (ABAC) column masking + row filtering for WarehousePG 6 or 7

Pure SQL / PL/pgSQL, no C extension. Run as a superuser in the target database.

Model
* role attributes  : ```abac.role_attribute(rolname, attr, value)```      e.g. fraud_team clearance=sensitive
* column tags      : ```abac.column_tag(schema, table, column, tag)``` e.g. customers.ssn -> pii:ssn
* masking policies : ```abac.masking_policy(tag -> required attr=value, mask_expr($1))```
* row policies     : ```abac.row_policy(tag -> filter_expr($1))```
* enforcement      : ```abac.protect(table)``` moves the base table to schema abac_protected and generates a security_barrier view under the original name that applies every policy bound to the table's tags. Re-run ```protect()``` after changing tags.

Attribute checks are emitted as uncorrelated sub-selects, so WHPG evaluates them once as
InitPlans on the coordinator and ships plain constants to the segments.

In [2]:
from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="w9-2-mdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

## Create table to be protected

Objective is to mask all but last 4 digits of column ```ssn``` and protect ```address1``` and ```address2``` using ```md5()```.

In [3]:
%%sql
CREATE TABLE customers (
    id         serial,
    first_name varchar(128),
    last_name  varchar(128),
    ssn        varchar(12),
    address1   varchar(128),
    address2   varchar(128),
    city       varchar(128),
    state      varchar(128),
    zip        varchar(10)
) DISTRIBUTED RANDOMLY;

INSERT INTO customers (first_name, last_name, ssn, address1, address2, city, state, zip) VALUES
    ('Ada',   'Lovelace', '123-45-6789', '1 Analytical Way', 'Suite 100', 'Denver',  'CO', '80202'),
    ('Alan',  'Turing',   '987-65-4321', '2 Enigma Rd',      NULL,        'Boulder', 'CO', '80301'),
    ('Grace', 'Hopper',   '555-12-9876', '3 Cobol Ct',       'Apt 7',     'Austin',  'TX', '78701');

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

3 rows affected.

++
||
++
++

## Create ABAC controls schema and objects

All tables are ```DISTRIBUTED REPLICATED``` so function executing on a segment has data locally.
    
```abac.role_attribute``` Attributes on roles, inherited through role membership

```abac.column_tag``` Tags on columns, keyed by the logical (queried) name of the table.

```abac.masking_policy``` Masking policy bound to a tag: clear text when caller has attr=value, otherwise mask_expr($1).

```abac.row_policy``` Row policy bound to a tag: a row is visible when filter_expr($1) is true.

In [4]:
%%sql
CREATE SCHEMA abac;             -- metadata + functions; usable by everyone
CREATE SCHEMA abac_protected;   -- base tables live here; no PUBLIC access
GRANT USAGE ON SCHEMA abac TO PUBLIC;

CREATE TABLE abac.role_attribute (
    rolname name NOT NULL,
    attr    text NOT NULL,
    value   text NOT NULL,
    PRIMARY KEY (rolname, attr, value)
) DISTRIBUTED REPLICATED;

GRANT SELECT ON abac.role_attribute TO PUBLIC;   -- has_attr()/attr_values() run as invoker

CREATE TABLE abac.column_tag (
    schema_name name NOT NULL,
    table_name  name NOT NULL,
    column_name name NOT NULL,
    tag         text NOT NULL,
    PRIMARY KEY (schema_name, table_name, column_name, tag)
) DISTRIBUTED REPLICATED;

CREATE TABLE abac.masking_policy (
    tag       text PRIMARY KEY,
    attr      text NOT NULL,
    value     text NOT NULL,
    mask_expr text NOT NULL
) DISTRIBUTED REPLICATED;

CREATE TABLE abac.row_policy (
    tag         text PRIMARY KEY,
    filter_expr text NOT NULL
) DISTRIBUTED REPLICATED;

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

++
||
++
++

```abac.has_attr()``` Does the current role (directly or via membership) hold attr=value?

```abac.attr_values()``` All values of attr held by the current role.

```abac.mask_last4('123-45-6789')``` -> '***-**-6789'

In [5]:
%%sql
CREATE FUNCTION abac.has_attr(p_attr text, p_value text) RETURNS boolean
LANGUAGE sql STABLE AS $$
    SELECT EXISTS (
        SELECT 1
          FROM abac.role_attribute ra
          JOIN pg_roles r ON r.rolname = ra.rolname
         WHERE ra.attr  = p_attr
           AND ra.value = p_value
           AND pg_has_role(current_user, r.oid, 'MEMBER'))
$$;

CREATE FUNCTION abac.attr_values(p_attr text) RETURNS text[]
LANGUAGE sql STABLE AS $$
    SELECT COALESCE(array_agg(ra.value), '{}'::text[])
      FROM abac.role_attribute ra
      JOIN pg_roles r ON r.rolname = ra.rolname
     WHERE ra.attr = p_attr
       AND pg_has_role(current_user, r.oid, 'MEMBER')
$$;

CREATE FUNCTION abac.mask_last4(v text) RETURNS text
LANGUAGE sql IMMUTABLE STRICT AS $$
    SELECT regexp_replace(left(v, greatest(length(v) - 4, 0)), '[[:alnum:]]', '*', 'g')
           || right(v, 4)
$$;

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

++
||
++
++

Attach a tag to a column.
Accepts the table before or after ```protect()``` (the view has the same columns).

In [6]:
%%sql
CREATE FUNCTION abac.tag_column(p_table regclass, p_column name, p_tag text) RETURNS void
LANGUAGE plpgsql AS $$
DECLARE
    v_schema name;
    v_name   name;
BEGIN
    SELECT n.nspname, c.relname INTO v_schema, v_name
      FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
     WHERE c.oid = p_table;

    IF v_schema = 'abac_protected' THEN
        RAISE EXCEPTION 'tag the logical table name, not the protected base table %', p_table;
    END IF;
    IF NOT EXISTS (SELECT 1 FROM pg_attribute
                    WHERE attrelid = p_table AND attname = p_column
                      AND attnum > 0 AND NOT attisdropped) THEN
        RAISE EXCEPTION 'column %.% does not exist', p_table, p_column;
    END IF;

    INSERT INTO abac.column_tag VALUES (v_schema, v_name, p_column, p_tag);
END
$$;

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

++
||
++
++

In [7]:
%%sql
-- Protect a table (first call) or regenerate its view (later calls). Returns the view SQL.
CREATE FUNCTION abac.protect(p_table regclass) RETURNS text
LANGUAGE plpgsql AS $$
DECLARE
    v_schema  name;
    v_name    name;
    v_kind    text;
    v_owner   name;
    v_base    regclass;
    v_first   boolean;
    v_cols    text[] := '{}';
    v_filters text[] := '{}';
    v_n       int;
    v_attr    text;
    v_value   text;
    v_expr    text;
    v_filter  text;
    v_sql     text;
    col       record;
    g         record;
BEGIN
    SELECT n.nspname, c.relname, c.relkind::text, pg_get_userbyid(c.relowner)
      INTO v_schema, v_name, v_kind, v_owner
      FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
     WHERE c.oid = p_table;

    IF v_schema = 'abac_protected' THEN
        RAISE EXCEPTION 'pass the logical table name, not the protected base table %', p_table;
    END IF;

    v_first := v_kind IN ('r', 'p');
    IF v_first THEN
        EXECUTE format('ALTER TABLE %I.%I SET SCHEMA abac_protected', v_schema, v_name);
    ELSIF v_kind <> 'v' OR to_regclass(format('abac_protected.%I', v_name)) IS NULL THEN
        RAISE EXCEPTION '% is neither a table nor an abac-managed view', p_table;
    END IF;
    v_base := format('abac_protected.%I', v_name)::regclass;

    FOR col IN
        SELECT a.attname, format_type(a.atttypid, a.atttypmod) AS typ
          FROM pg_attribute a
         WHERE a.attrelid = v_base AND a.attnum > 0 AND NOT a.attisdropped
         ORDER BY a.attnum
    LOOP
        -- masking: at most one masking policy per column
        SELECT count(*), min(mp.attr), min(mp.value), min(mp.mask_expr)
          INTO v_n, v_attr, v_value, v_expr
          FROM abac.column_tag ct
          JOIN abac.masking_policy mp ON mp.tag = ct.tag
         WHERE ct.schema_name = v_schema AND ct.table_name = v_name
           AND ct.column_name = col.attname;

        IF v_n > 1 THEN
            RAISE EXCEPTION 'column %.% has % masking policies; only one is allowed',
                            p_table, col.attname, v_n;
        ELSIF v_n = 1 THEN
            -- cast keeps the view column's declared type identical to the base column
            v_cols := v_cols || format(
                'CASE WHEN (SELECT abac.has_attr(%L, %L)) THEN %I ELSE (%s)::%s END AS %I',
                v_attr, v_value, col.attname,
                replace(v_expr, '$1', quote_ident(col.attname)), col.typ, col.attname);
        ELSE
            v_cols := v_cols || quote_ident(col.attname);
        END IF;

        -- row filters: all row policies on all tagged columns are ANDed
        SELECT string_agg('(' || replace(rp.filter_expr, '$1', quote_ident(col.attname)) || ')',
                          E'\n   AND ')
          INTO v_filter
          FROM abac.column_tag ct
          JOIN abac.row_policy rp ON rp.tag = ct.tag
         WHERE ct.schema_name = v_schema AND ct.table_name = v_name
           AND ct.column_name = col.attname;
        IF v_filter IS NOT NULL THEN
            v_filters := v_filters || v_filter;
        END IF;
    END LOOP;

    v_sql := format(E'CREATE OR REPLACE VIEW %I.%I WITH (security_barrier = true) AS\nSELECT %s\n  FROM %s%s',
                    v_schema, v_name,
                    array_to_string(v_cols, E',\n       '),
                    v_base,
                    CASE WHEN cardinality(v_filters) > 0
                         THEN E'\n WHERE ' || array_to_string(v_filters, E'\n   AND ')
                         ELSE '' END);
    EXECUTE v_sql;
    EXECUTE format('ALTER VIEW %I.%I OWNER TO %I', v_schema, v_name, v_owner);
    EXECUTE format('GRANT USAGE ON SCHEMA abac_protected TO %I', v_owner);

    IF v_first THEN
        -- move existing SELECT grants from the base table to the view
        FOR g IN
            SELECT DISTINCT CASE WHEN x.grantee = 0 THEN 'PUBLIC'
                                 ELSE quote_ident(pg_get_userbyid(x.grantee)) END AS grantee
              FROM pg_class c, aclexplode(c.relacl) x
             WHERE c.oid = v_base
               AND x.privilege_type = 'SELECT'
               AND x.grantee <> c.relowner
        LOOP
            EXECUTE format('REVOKE SELECT ON %s FROM %s', v_base, g.grantee);
            EXECUTE format('GRANT SELECT ON %I.%I TO %s', v_schema, v_name, g.grantee);
        END LOOP;
    END IF;

    RETURN v_sql;
END
$$;

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

++
||
++
++

Policies are bound to tags; tags are bound to columns

In [8]:
%%sql
INSERT INTO abac.masking_policy (tag, attr, value, mask_expr) VALUES
    ('pii:ssn',     'clearance', 'sensitive', 'abac.mask_last4($1)'),
    ('pii:address', 'clearance', 'mailers',   'md5($1)');

SELECT abac.tag_column('customers', 'ssn',      'pii:ssn');
SELECT abac.tag_column('customers', 'address1', 'pii:address');
SELECT abac.tag_column('customers', 'address2', 'pii:address');

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

2 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

tag_column
""


Roles and their attributes

In [9]:
%%sql
CREATE ROLE analyst    LOGIN;
CREATE ROLE fraud_team LOGIN;
CREATE ROLE mailroom   LOGIN;
GRANT SELECT ON customers TO analyst, fraud_team, mailroom;

INSERT INTO abac.role_attribute (rolname, attr, value) VALUES
    ('fraud_team', 'clearance', 'sensitive'),
    ('mailroom',   'clearance', 'mailers');

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

2 rows affected.

++
||
++
++

Enforce: customers -> abac_protected.customers, view public.customers takes its place

In [10]:
%%sql
SELECT abac.protect('customers');

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

1 rows affected.

protect
"CREATE OR REPLACE VIEW public.customers WITH (security_barrier = true) ASSELECT id, first_name, last_name, CASE WHEN (SELECT abac.has_attr('clearance', 'sensitive')) THEN ssn ELSE (abac.mask_last4(ssn))::character varying(12) END AS ssn, CASE WHEN (SELECT abac.has_attr('clearance', 'mailers')) THEN address1 ELSE (md5(address1))::character varying(128) END AS address1, CASE WHEN (SELECT abac.has_attr('clearance', 'mailers')) THEN address2 ELSE (md5(address2))::character varying(128) END AS address2, city, state, zip FROM abac_protected.customers"


Test (real users connect as themselves; SET ROLE is just convenient here)

In [11]:
%%sql
SET ROLE analyst;
SELECT id, ssn, address1, address2 FROM customers ORDER BY id;
SET ROLE fraud_team;
SELECT id, ssn, address1, address2 FROM customers ORDER BY id;
SET ROLE mailroom;
SELECT id, ssn, address1, address2 FROM customers ORDER BY id;
SELECT id FROM abac_protected.customers;    -- ERROR: permission denied for schema abac_protected
RESET ROLE;

Running query in 'postgresql://gpadmin@w9-2-mdw:5432/dev'

3 rows affected.

3 rows affected.

3 rows affected.

RuntimeError: (psycopg2.errors.InsufficientPrivilege) permission denied for schema abac_protected
LINE 1: SELECT id FROM abac_protected.customers;
                       ^

[SQL: SELECT id FROM abac_protected.customers;]
(Background on this error at: https://sqlalche.me/e/20/f405)


You should see the following output from above commands.

Mailroom sees full address and masked ssn.
```
[dev]=# set role mailroom ;
SET
(gspiegel@[local]) [dev]=> SELECT id, ssn, address1, address2 FROM customers ORDER BY id;
 id |     ssn     |     address1     | address2
----+-------------+------------------+-----------
  1 | ***-**-6789 | 1 Analytical Way | Suite 100
  2 | ***-**-4321 | 2 Enigma Rd      |
  3 | ***-**-9876 | 3 Cobol Ct       | Apt 7
(3 rows)
```
Analyst roleis permitted last 4 digits of ssn and hashed addresses.
```
[dev]=> set role analyst ;
SET
[dev]=> SELECT id, ssn, address1, address2 FROM customers ORDER BY id;
 id |     ssn     |             address1             |             address2
----+-------------+----------------------------------+----------------------------------
  1 | ***-**-6789 | a2bfc0e37c85133ee68b4c0c0de59a2f | 54ca312abad29c6065bb0f4e0d96f7cc
  2 | ***-**-4321 | 7e3624a23835338ac791f6fa2af976ef |
  3 | ***-**-9876 | 1028c98937ece84b18dce596ba802421 | e7d6aab28e9d148b8b21a61097d64ed9
(3 rows)
```
And fraud team is permitted full ssn & masked address columns.
```
[dev]=> set role fraud_team ;
SET
[dev]=> SELECT id, ssn, address1, address2 FROM customers ORDER BY id;
 id |     ssn     |             address1             |             address2
----+-------------+----------------------------------+----------------------------------
  1 | 123-45-6789 | a2bfc0e37c85133ee68b4c0c0de59a2f | 54ca312abad29c6065bb0f4e0d96f7cc
  2 | 987-65-4321 | 7e3624a23835338ac791f6fa2af976ef |
  3 | 555-12-9876 | 1028c98937ece84b18dce596ba802421 | e7d6aab28e9d148b8b21a61097d64ed9
(3 rows)
```

Row filtering on the same table. Tag "state" and re-run ```protect()```.
```protect()``` regenerates the view in place and all grants survive.

In [12]:
INSERT INTO abac.row_policy (tag, filter_expr) VALUES
    ('geo:state', $f$ $1 = ANY ((SELECT abac.attr_values('state'))::text[]) OR (SELECT abac.has_attr('state', '*')) $f$);

SELECT abac.tag_column('customers', 'state', 'geo:state');

INSERT INTO abac.role_attribute (rolname, attr, value) VALUES
    ('analyst',    'state', 'TX'),
    ('mailroom',   'state', 'CO'),
    ('fraud_team', 'state', '*');

SELECT abac.protect('customers');

SET ROLE analyst;
SELECT id, state, ssn FROM customers ORDER BY id;
SET ROLE mailroom;
SELECT id, state, ssn FROM customers ORDER BY id;
SET ROLE fraud_team;
SELECT id, state, ssn FROM customers ORDER BY id;
RESET ROLE;

SyntaxError: invalid syntax (231327160.py, line 1)

analyst role is limited to TX:
```
[dev]=# SET ROLE analyst;
SET
[dev]=# SELECT id, state, ssn FROM customers ORDER BY id;
 id | state |     ssn
----+-------+-------------
  3 | TX    | ***-**-9876
(1 row)
```
mailroom permitted TX and CO:
```
[dev]=> SET ROLE mailroom;
SET
[dev]=# SELECT id, state, ssn FROM customers ORDER BY id;
 id | state |     ssn
----+-------+-------------
  1 | CO    | ***-**-6789
  2 | CO    | ***-**-4321
(2 rows)
```
And fraud team is allowed all:
```
[dev]=> SET ROLE fraud_team;
SET
[dev]=# SELECT id, state, ssn FROM customers ORDER BY id;   -- all
 id | state |     ssn
----+-------+-------------
  1 | CO    | 123-45-6789
  2 | CO    | 987-65-4321
  3 | TX    | 555-12-9876
(3 rows)

## Alternative Row-Level Security

```CREATE POLICY``` is exactly the row half, and it is better than the ```WHERE``` baked into the view
because it also governs ```UPDATE```/```DELETE```, ```WITH CHECK``` on ```INSERT```, and ```COPY t TO```.

In [ ]:
ALTER TABLE abac_protected.customers ENABLE ROW LEVEL SECURITY;
ALTER TABLE abac_protected.customers FORCE  ROW LEVEL SECURITY;
CREATE POLICY geo_state ON abac_protected.customers FOR ALL TO PUBLIC
    USING (state = ANY ((SELECT abac.attr_values('state'))::text[])
           OR (SELECT abac.has_attr('state', '*')));

Through a view, RLS is evaluated as the view owner, not the caller.
Without ```FORCE```, the owner is exempt and mailroom saw all three rows. With ```FORCE```,
the policy applies and ```current_user``` inside ```has_attr()``` is still mailroom, so it filtered to CO.
Superusers bypass RLS unconditionally, so the table and view must be owned by a non-superuser
role such as ```data_owner```.
Same reason: use ```TO PUBLIC``` and put attribute logic in ```USING```;
a TO analyst clause would be matched against the owner.

```protect()``` can emit this ```CREATE POLICY``` instead of the ```WHERE``` clause with a five-line change.
RLS is in WarehousePG; check ```EXPLAIN``` for an ORCA fallback on RLS-enabled tables.

In [ ]:
REVOKE SELECT ON customers FROM analyst;
GRANT SELECT (id, first_name, last_name, city, state, zip) ON customers TO analyst;
-- analyst: SELECT id, state FROM customers  → works
--          SELECT id, ssn  FROM customers  → ERROR: permission denied for view customers
--          SELECT *        FROM customers  → ERROR (same)

That breaks ```SELECT *``` and any BI tool that introspects columns,
which is why masking needs the view (or a parse-tree rewrite in C).

The practical combination is ```CREATE POLICY``` on the base table for rows,
the generated view for masking, and column grants only where you want an outright deny.